# Document Question Answering System (RAG)
### A Retrieval-Augmented Generation pipeline over custom documents

This notebook builds a complete RAG system from the ground up: it loads a document (PDF or
text), splits it into chunks, embeds the chunks, stores them in a vector index, retrieves the
most relevant chunks for a question, and uses a language model to generate a grounded answer.

Everything runs locally with free, open-source models. No API key required. Swap in your own
PDF (resume, notes, research paper, book) by changing one variable, see the last section.


## Objectives
- Understand how retrieval and generation combine in a RAG system
- Build every pipeline stage by hand: ingestion, chunking, embedding, indexing, retrieval, generation
- Answer questions grounded in a custom document instead of the model's own memorised knowledge
- Experiment with chunking, hybrid search, and re-ranking to see what actually improves retrieval


## Key Concepts

| Stage | What it does |
|---|---|
| Retrieval | Finds the most relevant chunks of text using embedding similarity |
| Augmentation | Inserts those chunks into the model's prompt as context |
| Generation | The language model answers using only that context |

The point of RAG is that the model is not answering from what it memorised during training.
It is answering from what you hand it, right now. That is what makes it useful for private,
recent, or niche documents a general model has never seen.


## Pipeline

1. **Document Ingestion** - load a PDF or text file into raw text
2. **Chunking** - split the raw text into overlapping word windows
3. **Embedding** - convert each chunk into a vector with a sentence-transformer
4. **Vector Store** - index the vectors with FAISS for fast similarity search
5. **Retrieval** - embed the question, pull back the top-k most similar chunks
6. **Generation** - feed the question and retrieved chunks to an LLM, get a grounded answer


## Setup

Run this once. CPU only, no GPU or API key needed.


In [1]:
!pip install -q pypdf reportlab sentence-transformers faiss-cpu transformers torch rank_bm25 2>/dev/null \
  || pip install -q --break-system-packages pypdf reportlab sentence-transformers faiss-cpu transformers torch rank_bm25

In [2]:
import re
import numpy as np
import faiss
import torch
from pypdf import PdfReader
from reportlab.lib.pagesizes import letter
from reportlab.lib.units import inch
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rank_bm25 import BM25Okapi

np.random.seed(42)
print("Setup complete. GPU available:", torch.cuda.is_available())

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete. GPU available: False


## Step 1: Document Ingestion

Loads `.pdf` or `.txt` files into plain text. This covers the two formats mentioned in the
brief (notes, resumes, papers and books are almost always one of these two under the hood).


In [3]:
def load_pdf(path: str) -> str:
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text

def load_txt(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def load_document(path: str) -> str:
    if path.lower().endswith(".pdf"):
        return load_pdf(path)
    elif path.lower().endswith(".txt"):
        return load_txt(path)
    raise ValueError("Unsupported file type. Use a .pdf or .txt file.")

## Demo document

No PDF on hand? This cell generates a small fictional company handbook
("GreenLeaf Solar Solutions") with specific invented facts: founding year, employee count,
warranty terms, support hours. A general-purpose language model has no way of knowing these
numbers from training, so if the system answers them correctly, that is proof the answer came
from retrieval and not from the model guessing.

**To use your own document instead:** upload your resume, notes, or paper next to this
notebook, set `DOCUMENT_PATH` to its filename, and skip the `build_sample_pdf(...)` call below.


In [4]:
def build_sample_pdf(path: str) -> None:
    sections = [
        ("GreenLeaf Solar Solutions - Company Handbook", None),
        ("1. Company Overview",
         "GreenLeaf Solar Solutions was founded in 2014 in Manchester, United Kingdom, by a team "
         "of three renewable energy engineers. The company designs, manufactures and installs "
         "residential and commercial solar panel systems across the UK and Ireland. As of 2026, "
         "GreenLeaf employs 342 people across five regional offices located in Manchester, "
         "Bristol, Glasgow, Dublin and Leeds. The company mission is to make clean, affordable "
         "solar energy accessible to every household within a decade."),
        ("2. Product Range",
         "GreenLeaf currently manufactures three solar panel models. The SolarMax 350 is the "
         "entry level residential panel, rated at 350 watts with a conversion efficiency of 20.1 "
         "percent. The SolarMax 400 is the mid range model, rated at 400 watts with an efficiency "
         "of 21.4 percent, and is GreenLeaf's best selling product, accounting for 61 percent of "
         "total sales in 2025. The SolarMax 500 Pro is the premium commercial grade panel, rated "
         "at 500 watts with an efficiency of 22.8 percent, designed for large rooftop installations."),
        ("3. Warranty Policy",
         "All GreenLeaf panels come with a standard 25 year performance warranty, guaranteeing at "
         "least 84 percent of original output at the end of the warranty period. The SolarMax 500 "
         "Pro carries an extended 30 year warranty due to its commercial grade components. "
         "Inverters supplied by GreenLeaf carry a separate 12 year warranty, extendable to 20 "
         "years for an additional fee of 180 pounds."),
        ("4. Installation Process",
         "A typical residential installation follows five steps: site survey, system design, "
         "permitting and grid approval, physical installation, and final inspection. The average "
         "residential installation takes 11 days from the initial site survey to grid connection, "
         "though this can extend to 6 weeks in areas requiring additional grid capacity approval "
         "from the local Distribution Network Operator."),
        ("5. Customer Support",
         "GreenLeaf's customer support team is available Monday to Saturday, from 8 AM to 7 PM. "
         "The average first response time for support tickets is 4 hours during weekdays. "
         "Customers can reach support via phone, email, or the GreenLeaf mobile app, which also "
         "allows real time monitoring of system output."),
        ("6. Returns and Cancellation",
         "Customers may cancel an installation contract within 14 days of signing without "
         "penalty, in line with UK consumer protection regulations. After installation has begun, "
         "cancellation incurs a fee equal to 15 percent of the total contract value to cover "
         "materials and labour already committed."),
        ("7. Environmental Impact",
         "In 2025, GreenLeaf's installed systems collectively generated an estimated 96,000 "
         "megawatt hours of clean electricity, offsetting approximately 21,000 tonnes of carbon "
         "dioxide emissions. The company aims to reach 150,000 installations nationwide by 2028."),
        ("8. Careers and Culture",
         "GreenLeaf hires for roles across engineering, installation, sales and customer support. "
         "New graduate engineers typically join through a structured 6 month rotational "
         "programme covering design, installation supervision, and quality assurance. The company "
         "was named one of the UK's Top 50 Green Employers in 2025."),
        ("9. Certifications",
         "GreenLeaf holds MCS, the Microgeneration Certification Scheme accreditation, for all "
         "its panel and inverter products, and is a registered member of the Renewable Energy "
         "Consumer Code, known as RECC."),
    ]

    doc = SimpleDocTemplate(path, pagesize=letter, topMargin=0.8 * inch, bottomMargin=0.8 * inch)
    styles = getSampleStyleSheet()
    title_style = ParagraphStyle("TitleStyle", parent=styles["Title"], fontSize=16)
    heading_style = ParagraphStyle("HeadingStyle", parent=styles["Heading2"], spaceBefore=12, spaceAfter=6)
    body_style = ParagraphStyle("BodyStyle", parent=styles["Normal"], fontSize=10.5, leading=15)

    story = []
    for heading, body in sections:
        if body is None:
            story.append(Paragraph(heading, title_style))
            story.append(Spacer(1, 14))
        else:
            story.append(Paragraph(heading, heading_style))
            story.append(Paragraph(body, body_style))
    doc.build(story)


DOCUMENT_PATH = "sample_document.pdf"
build_sample_pdf(DOCUMENT_PATH)

raw_text = load_document(DOCUMENT_PATH)
print(f"Loaded {len(raw_text)} characters from {DOCUMENT_PATH}\n")
print(raw_text[:500], "...")

Loaded 3376 characters from sample_document.pdf

GreenLeaf Solar Solutions - Company Handbook
1. Company Overview
GreenLeaf Solar Solutions was founded in 2014 in Manchester, United Kingdom, by a team of
three renewable energy engineers. The company designs, manufactures and installs residential
and commercial solar panel systems across the UK and Ireland. As of 2026, GreenLeaf employs
342 people across five regional offices located in Manchester, Bristol, Glasgow, Dublin and Leeds.
The company mission is to make clean, affordable solar energy ...


## Step 2: Text Chunking

Splits the document into overlapping windows of words. The overlap stops a sentence that
straddles a chunk boundary from getting cut in half and losing the answer it contains.


In [5]:
def chunk_text(text: str, chunk_size: int = 90, chunk_overlap: int = 20):
    text = re.sub(r"\s+", " ", text).strip()
    words = text.split(" ")
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        if end >= len(words):
            break
        start = end - chunk_overlap
    return chunks


chunks = chunk_text(raw_text, chunk_size=90, chunk_overlap=20)
avg_len = sum(len(c.split()) for c in chunks) // len(chunks)
print(f"Split into {len(chunks)} chunks (avg {avg_len} words each)\n")
for i, c in enumerate(chunks):
    print(f"[{i}] {c[:90]}...")

Split into 7 chunks (avg 89 words each)

[0] GreenLeaf Solar Solutions - Company Handbook 1. Company Overview GreenLeaf Solar Solutions...
[1] clean, affordable solar energy accessible to every household within a decade. 2. Product R...
[2] total sales in 2025. The SolarMax 500 Pro is the premium commercial grade panel, rated at ...
[3] its commercial grade components. Inverters supplied by GreenLeaf carry a separate 12 year ...
[4] in areas requiring additional grid capacity approval from the local Distribution Network O...
[5] may cancel an installation contract within 14 days of signing without penalty, in line wit...
[6] company aims to reach 150,000 installations nationwide by 2028. 8. Careers and Culture Gre...


## Step 3: Embedding Creation

`all-MiniLM-L6-v2` turns each chunk into a 384-dimensional vector. Chunks with similar
meaning end up close together in that vector space, even when they do not share exact words.


In [6]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = embedder.encode(chunks, convert_to_numpy=True, show_progress_bar=True).astype("float32")
print("Embedding matrix shape:", chunk_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 28684.24it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.88it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.83it/s]

Embedding matrix shape: (7, 384)


## Step 4: Vector Database

FAISS stores the vectors and runs the similarity search. Vectors are L2-normalised so that
inner product similarity equals cosine similarity.


In [7]:
dimension = chunk_embeddings.shape[1]
faiss.normalize_L2(chunk_embeddings)
index = faiss.IndexFlatIP(dimension)
index.add(chunk_embeddings)
print(f"Indexed {index.ntotal} chunks of dimension {dimension}")

Indexed 7 chunks of dimension 384


## Step 5 & 6: Query Processing and Context Retrieval

Embed the question with the same model used for the chunks, then pull back whichever chunks
sit closest to it in vector space.


In [8]:
def retrieve(query: str, top_k: int = 3):
    q_vec = embedder.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_vec)
    scores, idxs = index.search(q_vec, top_k)
    return [(chunks[i], float(scores[0][j])) for j, i in enumerate(idxs[0])]


for chunk, score in retrieve("What are the customer support hours?", top_k=3):
    print(f"{score:.3f} | {chunk[:100]}...")

0.550 | in areas requiring additional grid capacity approval from the local Distribution Network Operator. 5...
0.320 | its commercial grade components. Inverters supplied by GreenLeaf carry a separate 12 year warranty, ...
0.231 | may cancel an installation contract within 14 days of signing without penalty, in line with UK consu...


## Step 7: Answer Generation

`flan-t5-base` reads the retrieved chunks as context and answers only from them. The prompt
explicitly tells it to say so when the answer is not in the context. That instruction matters:
an ungrounded model will happily guess rather than admit it does not know.


In [9]:
gen_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
gen_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")


def generate_answer(query: str, retrieved_chunks, max_new_tokens: int = 80) -> str:
    context = "\n\n".join(chunk for chunk, _ in retrieved_chunks)
    prompt = (
        "Answer the question using ONLY the information in the context below. "
        "If the answer cannot be found in the context, respond exactly with: "
        "Not mentioned in the document.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\nAnswer:"
    )
    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    output_ids = gen_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return gen_tokenizer.decode(output_ids[0], skip_special_tokens=True)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 38128.81it/s]


[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


## Putting it together

One function that runs retrieval and generation, and shows which chunks the answer is
grounded in, so you can sanity-check it yourself instead of taking the answer on faith.


In [10]:
def rag_query(query: str, top_k: int = 3, verbose: bool = True):
    retrieved = retrieve(query, top_k=top_k)
    answer = generate_answer(query, retrieved)
    if verbose:
        print(f"Q: {query}")
        print(f"A: {answer}\n")
        print("Sources used:")
        for chunk, score in retrieved:
            print(f"  ({score:.3f}) {chunk[:90]}...")
    return answer, retrieved

## Example flow

The exact question from the brief ("What is the main idea of the document?"), a few
questions that check specific facts, and one question with no answer in the document at all,
to confirm the refusal behaviour actually works instead of hallucinating.


In [11]:
demo_questions = [
    "What is the main idea of this document?",
    "What is the warranty period for the SolarMax 400 panel?",
    "What are the customer support hours?",
    "What happens if I cancel after installation has already started?",
    "What is GreenLeaf's stock price?",
]

for q in demo_questions:
    rag_query(q)
    print("-" * 80)

Q: What is the main idea of this document?
A: The company mission is to make clean, affordable solar energy accessible to every household within a decade.

Sources used:
  (0.075) company aims to reach 150,000 installations nationwide by 2028. 8. Careers and Culture Gre...
  (0.024) GreenLeaf Solar Solutions - Company Handbook 1. Company Overview GreenLeaf Solar Solutions...
  (0.008) may cancel an installation contract within 14 days of signing without penalty, in line wit...
--------------------------------------------------------------------------------


Q: What is the warranty period for the SolarMax 400 panel?
A: 25 years

Sources used:
  (0.697) total sales in 2025. The SolarMax 500 Pro is the premium commercial grade panel, rated at ...
  (0.462) clean, affordable solar energy accessible to every household within a decade. 2. Product R...
  (0.433) its commercial grade components. Inverters supplied by GreenLeaf carry a separate 12 year ...
--------------------------------------------------------------------------------


Q: What are the customer support hours?
A: Monday to Saturday, from 8 AM to 7 PM

Sources used:
  (0.550) in areas requiring additional grid capacity approval from the local Distribution Network O...
  (0.320) its commercial grade components. Inverters supplied by GreenLeaf carry a separate 12 year ...
  (0.231) may cancel an installation contract within 14 days of signing without penalty, in line wit...
--------------------------------------------------------------------------------


Q: What happens if I cancel after installation has already started?
A: cancellation incurs a fee equal to 15 percent of the total contract value to cover materials and labour already committed

Sources used:
  (0.426) may cancel an installation contract within 14 days of signing without penalty, in line wit...
  (0.230) in areas requiring additional grid capacity approval from the local Distribution Network O...
  (0.215) its commercial grade components. Inverters supplied by GreenLeaf carry a separate 12 year ...
--------------------------------------------------------------------------------


Q: What is GreenLeaf's stock price?
A: Not mentioned in the document.

Sources used:
  (0.455) may cancel an installation contract within 14 days of signing without penalty, in line wit...
  (0.435) GreenLeaf Solar Solutions - Company Handbook 1. Company Overview GreenLeaf Solar Solutions...
  (0.423) in areas requiring additional grid capacity approval from the local Distribution Network O...
--------------------------------------------------------------------------------


## Try your own question

Edit the string below and re-run the cell.


In [12]:
your_question = "How many people work at GreenLeaf and across how many offices?"
rag_query(your_question)

Q: How many people work at GreenLeaf and across how many offices?
A: 342

Sources used:
  (0.549) company aims to reach 150,000 installations nationwide by 2028. 8. Careers and Culture Gre...
  (0.536) GreenLeaf Solar Solutions - Company Handbook 1. Company Overview GreenLeaf Solar Solutions...
  (0.481) may cancel an installation contract within 14 days of signing without penalty, in line wit...


('342',
 [("company aims to reach 150,000 installations nationwide by 2028. 8. Careers and Culture GreenLeaf hires for roles across engineering, installation, sales and customer support. New graduate engineers typically join through a structured 6 month rotational programme covering design, installation supervision, and quality assurance. The company was named one of the UK's Top 50 Green Employers in 2025. 9. Certifications GreenLeaf holds MCS, the Microgeneration Certification Scheme accreditation, for all its panel and inverter products, and is a registered member of the Renewable Energy Consumer Code, known as RECC.",
   0.5486757755279541),
  ('GreenLeaf Solar Solutions - Company Handbook 1. Company Overview GreenLeaf Solar Solutions was founded in 2014 in Manchester, United Kingdom, by a team of three renewable energy engineers. The company designs, manufactures and installs residential and commercial solar panel systems across the UK and Ireland. As of 2026, GreenLeaf employs 34

## Improvements & Experiments

Three things that reliably improve retrieval quality in real systems, demonstrated here on
the same document.


### Experiment 1: Chunk size

Smaller chunks are more precise but lose surrounding context. Larger chunks keep context but
dilute the match, since one vector now represents several ideas at once. There is no
universally correct number, it depends on how the source document is structured.


In [13]:
for size, overlap in [(30, 5), (90, 20), (200, 30)]:
    test_chunks = chunk_text(raw_text, chunk_size=size, chunk_overlap=overlap)
    print(f"chunk_size={size:>4}  ->  {len(test_chunks):>2} chunks, ~{size} words each")

chunk_size=  30  ->  21 chunks, ~30 words each
chunk_size=  90  ->   7 chunks, ~90 words each
chunk_size= 200  ->   3 chunks, ~200 words each


### Experiment 2: Hybrid search (keyword + vector)

Vector search occasionally misses an exact keyword match, such as a specific model number or
an uncommon term, because it optimises for meaning rather than exact wording. BM25 is a
classic keyword-ranking algorithm. Blending the two scores catches what either one misses
alone.


In [14]:
tokenized_corpus = [c.lower().split() for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)


def _minmax(scores):
    scores = np.array(scores, dtype="float32")
    spread = scores.max() - scores.min()
    return np.zeros_like(scores) if spread < 1e-9 else (scores - scores.min()) / spread


def hybrid_retrieve(query: str, top_k: int = 3, alpha: float = 0.5):
    q_vec = embedder.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_vec)
    vec_scores, vec_idx = index.search(q_vec, len(chunks))
    vec_full = np.zeros(len(chunks), dtype="float32")
    for score, idx in zip(vec_scores[0], vec_idx[0]):
        vec_full[idx] = score
    bm25_full = bm25.get_scores(query.lower().split())
    combined = alpha * _minmax(vec_full) + (1 - alpha) * _minmax(bm25_full)
    top_idx = np.argsort(combined)[::-1][:top_k]
    return [(chunks[i], float(combined[i])) for i in top_idx]


query = "How long does installation take and what fee applies if I cancel it midway?"
print("Vector only:")
for c, s in retrieve(query, top_k=3):
    print(f"  {s:.3f} | {c[:90]}...")

print("\nHybrid (BM25 + vector):")
for c, s in hybrid_retrieve(query, top_k=3):
    print(f"  {s:.3f} | {c[:90]}...")

Vector only:
  0.493 | may cancel an installation contract within 14 days of signing without penalty, in line wit...
  0.408 | its commercial grade components. Inverters supplied by GreenLeaf carry a separate 12 year ...
  0.323 | in areas requiring additional grid capacity approval from the local Distribution Network O...

Hybrid (BM25 + vector):
  1.000 | may cancel an installation contract within 14 days of signing without penalty, in line wit...
  0.759 | its commercial grade components. Inverters supplied by GreenLeaf carry a separate 12 year ...
  0.611 | in areas requiring additional grid capacity approval from the local Distribution Network O...


### Experiment 3: Cross-encoder re-ranking

The bi-encoder used for retrieval scores the query and each chunk separately and compares the
two vectors, which is fast but loses some nuance. A cross-encoder reads the query and chunk
together and scores them jointly, which is slower but sharper. The common pattern: retrieve a
wider net with the fast bi-encoder, then re-rank just the top handful with the slower
cross-encoder.


In [15]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def rerank(query: str, candidates, top_k: int = 3):
    pairs = [(query, c) for c, _ in candidates]
    scores = cross_encoder.predict(pairs)
    order = np.argsort(scores)[::-1][:top_k]
    return [(candidates[i][0], float(scores[i])) for i in order]


wide_candidates = retrieve(query, top_k=5)
print("Before re-ranking (bi-encoder order):")
for c, s in wide_candidates[:3]:
    print(f"  {s:.3f} | {c[:90]}...")

print("\nAfter re-ranking (cross-encoder order):")
for c, s in rerank(query, wide_candidates, top_k=3):
    print(f"  {s:.3f} | {c[:90]}...")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 26788.44it/s]

Before re-ranking (bi-encoder order):
  0.493 | may cancel an installation contract within 14 days of signing without penalty, in line wit...
  0.408 | its commercial grade components. Inverters supplied by GreenLeaf carry a separate 12 year ...
  0.323 | in areas requiring additional grid capacity approval from the local Distribution Network O...

After re-ranking (cross-encoder order):
  0.498 | may cancel an installation contract within 14 days of signing without penalty, in line wit...
  -1.796 | in areas requiring additional grid capacity approval from the local Distribution Network O...
  -3.928 | its commercial grade components. Inverters supplied by GreenLeaf carry a separate 12 year ...


### Experiment 4: Swapping in a different language model

`flan-t5-base` is small, free, and fine for a demo, but it is not the strongest reasoner
available. For better answer quality, point `generate_answer` at an API model instead. This
needs your own API key and is not executed in this notebook, it is here to show the shape of
the change.


In [16]:
USE_API_MODEL = False  # flip to True and add your own key to actually try this

if USE_API_MODEL:
    import anthropic

    client = anthropic.Anthropic(api_key="YOUR_API_KEY")

    def generate_answer_claude(query, retrieved_chunks):
        context = "\n\n".join(chunk for chunk, _ in retrieved_chunks)
        message = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=300,
            messages=[{
                "role": "user",
                "content": (
                    f"Context:\n{context}\n\n"
                    f"Question: {query}\n"
                    "Answer using only the context above. If it is not there, say so."
                ),
            }],
        )
        return message.content[0].text

## Key Learnings

- Retrieval quality controls answer quality more than model size does. A well-retrieved
  chunk with a small model beats a large model given the wrong chunk.
- Chunking is a real design decision, not boilerplate. The wrong chunk size splits relevant
  information away from the question that needs it.
- Grounded generation needs an explicit instruction to defer to the context and to say when
  it does not know. Without that instruction, the model guesses from its own training data
  instead of admitting the document does not cover it.
- Hybrid search and re-ranking are cheap to add and catch real failure cases that plain
  vector search misses on their own.


## Conclusion

This pipeline answers questions about a document a general-purpose model has never seen,
using only open-source components and free local models. The same six stages (ingest, chunk,
embed, index, retrieve, generate) are what production systems run under the hood for
chatbots, internal knowledge search, and document assistants. Swap in your own PDF, tune the
chunk size, and the same code keeps working.
